In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


# -----------------------------
# 1. Определяем state графа
# -----------------------------
class ChatState(TypedDict, total=False):
    user_message: str
    intent: Literal["faq", "booking"]
    answer: str
    booking_status: str


# -----------------------------
# 2. Router: понимаем, что хочет пользователь
# -----------------------------
def detect_intent(state: ChatState):
    text = state["user_message"].lower()

    if "book" in text or "reserve" in text:
        return {"intent": "booking"}

    return {"intent": "faq"}


# -----------------------------
# 3. Узел FAQ: обычный ответ
# -----------------------------
def faq_node(state: ChatState):
    text = state["user_message"].lower()

    if "price" in text:
        return {"answer": "Parking price is 5 EUR per hour."}

    if "hours" in text:
        return {"answer": "Parking works 24/7."}

    return {"answer": "I can answer about parking price, hours, and reservations."}


# -----------------------------
# 4. Узел бронирования
#    Здесь специально делаем interrupt
# -----------------------------
def booking_node(state: ChatState):
    decision = interrupt(
        {
            "type": "admin_approval",
            "message": "Booking request is waiting for admin approval"
        }
    )

    if decision == "approve":
        return {
            "booking_status": "approved",
            "answer": "Your booking has been approved."
        }

    return {
        "booking_status": "rejected",
        "answer": "Your booking has been rejected."
    }


# -----------------------------
# 5. Условный роутинг
# -----------------------------
def route_after_intent(state: ChatState):
    if state["intent"] == "booking":
        return "booking_node"
    return "faq_node"

In [2]:

# -----------------------------
# 6. Собираем граф
# -----------------------------
builder = StateGraph(ChatState)

builder.add_node("detect_intent", detect_intent)
builder.add_node("faq_node", faq_node)
builder.add_node("booking_node", booking_node)

builder.add_edge(START, "detect_intent")
builder.add_conditional_edges("detect_intent", route_after_intent)
builder.add_edge("faq_node", END)
builder.add_edge("booking_node", END)

graph = builder.compile(checkpointer=InMemorySaver())


# -----------------------------
# 7. Конфиги для разных thread_id
# -----------------------------
booking_thread = {"configurable": {"thread_id": "booking-1"}}
faq_thread = {"configurable": {"thread_id": "faq-1"}}

In [3]:
# -----------------------------
# 8. Сообщение 1:
#    пользователь начинает бронь
# -----------------------------
print("\n--- MESSAGE 1: booking request ---")
result1 = graph.invoke(
    {"user_message": "I want to book a parking spot"},
    config=booking_thread,
)
print(result1)

# Здесь будет interrupt, в ответе ты увидишь __interrupt__


--- MESSAGE 1: booking request ---
{'user_message': 'I want to book a parking spot', 'intent': 'booking', '__interrupt__': [Interrupt(value={'type': 'admin_approval', 'message': 'Booking request is waiting for admin approval'}, id='a802f88072036401183d454313246ded')]}


In [ ]:
# -----------------------------
# 9. Сообщение 2:
#    пока бронь ждет approve/reject,
#    пользователь задает обычный вопрос
# -----------------------------
print("\n--- MESSAGE 2: normal FAQ while booking is paused ---")
result2 = graph.invoke(
    {"user_message": "What is the price?"},
    config=faq_thread,
)
print(result2)

# Здесь бот спокойно ответит на FAQ

In [ ]:

# -----------------------------
# 10. Позже админ аппрувит бронь
#     Возобновляем именно booking thread
# -----------------------------
print("\n--- RESUME BOOKING THREAD ---")
result3 = graph.invoke(
    Command(resume="approve"),
    config=booking_thread,
)
print(result3)